# Multi-Agent Trading Simulation (KERNEL Framework)

This notebook simulates the multi-agent trading system by iterating through historical CSV data bar-by-bar.

## **IMPORTANT: Uses Shared Services (Identical to Production)**

This notebook imports the **exact same analysis logic** as the production agents via shared services:

| Shared Service | Production Agent | What It Does |
|----------------|------------------|--------------|
| `services/analysis/market_structure.py` | All agents | Swing points, Order Blocks, FVGs |
| `services/analysis/visual_analysis.py` | `visual_analyst_agent.py` | LLM Vision chart analysis |
| `services/analysis/meta_decision.py` | `meta_agent.py` | Weighted voting, decision logic |
| `services/timeframe_cascade.py` | `technicalanalyst_agent.py` | Multi-timeframe cascade |

**When you modify a shared service, BOTH production AND backtest use the new logic!**

## Agent Pipeline
1. **Technical Analyst** - Rule-based multi-timeframe analysis (`services/timeframe_cascade.py`)
2. **Visual Analyst** - **LLM Vision API** for chart pattern recognition (`services/analysis/visual_analysis.py`)
3. **Meta-Agent** - **LLM-based** strategic coordinator (`services/analysis/meta_decision.py`)
4. **Risk Manager** - Rule-based trade validation and position sizing
5. **Execution Agent** - Order placement (simulated for backtest)
6. **Trade Manager** - Rule-based position monitoring

## Key Features
- **Identical logic**: Backtest uses same services as production server
- Real LLM calls (Gemini/Claude) for Visual Analyst and Meta-Agent
- No look-ahead bias (processes data bar-by-bar)
- Order Blocks, FVGs, Swing Points detection
- Full agent communication logging
- P&L tracking

In [1]:
# Cell 1: Imports and Setup
import sys
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, field
import json
import io
import base64
import logging
import matplotlib.pyplot as plt
import mplfinance as mpf
from PIL import Image

# Add backend to path for imports
backend_path = os.path.join(os.getcwd(), '..', 'backend')
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

# Load environment variables
from dotenv import load_dotenv
load_dotenv(os.path.join(backend_path, '.env'))

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

print("✓ Imports loaded")
print(f"  Backend path: {backend_path}")
print(f"  LLM Provider: {os.getenv('LLM_PROVIDER', 'gemini')}")
print(f"  Google API Key: {'✓ Set' if os.getenv('GOOGLE_API_KEY') else '✗ Missing'}")
print(f"  Anthropic API Key: {'✓ Set' if os.getenv('ANTHROPIC_API_KEY') else '✗ Missing'}")

✓ Imports loaded
  Backend path: /Users/x/Desktop/algo_project/notebooks/../backend
  LLM Provider: gemini
  Google API Key: ✓ Set
  Anthropic API Key: ✓ Set


In [2]:
# Cell 2: Import Backend Agents and LLM Client

# Import from actual backend
from config.llm_provider import get_llm_client, get_provider_info
from agents.base_agent import AgentType, MessageType, Message

# Initialize LLM client (will use GOOGLE_API_KEY or ANTHROPIC_API_KEY)
try:
    llm_client = get_llm_client(force_new=True)
    provider_info = get_provider_info()
    print(f"✓ LLM Client initialized")
    print(f"  Provider: {provider_info['provider']}")
    print(f"  Model: {provider_info['model']}")
except Exception as e:
    print(f"✗ Failed to initialize LLM client: {e}")
    print("  Set GOOGLE_API_KEY or ANTHROPIC_API_KEY in your environment")
    llm_client = None

/Users/x/Desktop/algo_project/trade_bot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/x/Desktop/algo_project/notebooks/../backend/config/llm_provider.py:96: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
✓ Gemini client initialized (model: gemini-2.0-flash)
✓ LLM Provider: gemini (gemini-2.0-flash)


✓ LLM Client initialized
  Provider: gemini
  Model: gemini-2.0-flash


In [3]:
# Cell 3: Market Structure Detection (SHARED SERVICE)
# Imported from backend/services/analysis/market_structure.py
# Changes to that file will be reflected here!

from services.analysis.market_structure import (
    MarketStructureAnalyzer,
    SwingPoint,
    OrderBlock,
    FairValueGap,
    create_market_structure_analyzer
)

# Create alias for backwards compatibility
MarketStructure = MarketStructureAnalyzer

print("✓ Market Structure imported from shared service")
print("  Source: backend/services/analysis/market_structure.py")

✓ Market Structure imported from shared service
  Source: backend/services/analysis/market_structure.py


In [4]:
# Cell 4: Multi-Timeframe Chart Generator for Visual Analyst

# Configuration: Bars per timeframe (matching production chart_generator.py)
TIMEFRAME_CONFIG = {
    '1min': {'bars': 200, 'period': 1, 'description': '~3.3 hours'},
    '5min': {'bars': 200, 'period': 5, 'description': '~16.7 hours'},
    '15min': {'bars': 200, 'period': 15, 'description': '~50 hours (2 days)'},
    '1H': {'bars': 200, 'period': 60, 'description': '~8.3 days'}
}

def aggregate_to_timeframe(df: pd.DataFrame, period_minutes: int) -> pd.DataFrame:
    """
    Aggregate 1-minute data to higher timeframe
    
    Args:
        df: DataFrame with 1-min OHLCV data
        period_minutes: Target period in minutes (5, 15, 60, etc.)
    
    Returns:
        Aggregated DataFrame
    """
    df = df.copy()
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df = df.set_index('Timestamp')
    
    # Resample to target timeframe
    resampled = df.resample(f'{period_minutes}min').agg({
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum'
    }).dropna()
    
    return resampled.reset_index()


def generate_chart_image(df: pd.DataFrame, idx: int, lookback: int = 200, 
                         title: str = 'Chart', timeframe: str = '1min') -> bytes:
    """
    Generate a candlestick chart image
    
    Args:
        df: DataFrame with OHLCV data
        idx: Current bar index (or -1 for end of data)
        lookback: Number of bars to include
        title: Chart title
        timeframe: Timeframe label for title
    
    Returns:
        bytes: PNG image data
    """
    if idx == -1:
        idx = len(df) - 1
    
    start_idx = max(0, idx - lookback + 1)
    plot_df = df.iloc[start_idx:idx+1].copy()
    
    # Set index for mplfinance
    if 'Timestamp' in plot_df.columns:
        plot_df = plot_df.set_index('Timestamp')
    
    # Create chart
    buf = io.BytesIO()
    
    # Calculate y-axis range for auto-zoom
    high_max = plot_df['High'].max()
    low_min = plot_df['Low'].min()
    price_range = high_max - low_min
    padding = price_range * 0.05
    
    mpf.plot(
        plot_df,
        type='candle',
        style='charles',
        title=f'{title} - {timeframe} ({len(plot_df)} bars)',
        ylabel='Price ($)',
        volume=False,  # Gold has no volume
        figsize=(14, 8),
        ylim=(low_min - padding, high_max + padding),
        tight_layout=True,
        savefig=dict(fname=buf, format='png', dpi=100, bbox_inches='tight')
    )
    
    buf.seek(0)
    return buf.read()


def generate_multi_timeframe_charts(df: pd.DataFrame, idx: int, symbol: str = 'USGOLD') -> Dict[str, bytes]:
    """
    Generate charts for all configured timeframes
    
    Args:
        df: 1-minute DataFrame
        idx: Current bar index in 1-min data
        symbol: Symbol name for title
    
    Returns:
        Dict mapping timeframe to PNG bytes
    """
    charts = {}
    
    # Get data up to current index (no look-ahead)
    data_slice = df.iloc[:idx+1].copy()
    
    for tf_name, tf_config in TIMEFRAME_CONFIG.items():
        period = tf_config['period']
        bars = tf_config['bars']
        
        try:
            if period == 1:
                # 1-minute: use directly
                tf_data = data_slice.copy()
            else:
                # Aggregate to higher timeframe
                tf_data = aggregate_to_timeframe(data_slice, period)
            
            if len(tf_data) < 20:
                print(f"    ⚠️ {tf_name}: Not enough data ({len(tf_data)} bars, need 20+)")
                continue
            
            # Generate chart
            chart_bytes = generate_chart_image(
                tf_data, 
                idx=-1,  # Use end of aggregated data
                lookback=bars,
                title=symbol,
                timeframe=tf_name
            )
            charts[tf_name] = chart_bytes
            print(f"    ✓ {tf_name}: {min(len(tf_data), bars)} bars ({tf_config['description']})")
            
        except Exception as e:
            print(f"    ✗ {tf_name}: Error - {e}")
    
    return charts


def create_multi_timeframe_grid(charts: Dict[str, bytes]) -> bytes:
    """
    Combine multiple chart images into a 2x2 grid
    
    Args:
        charts: Dict mapping timeframe to PNG bytes
    
    Returns:
        Combined PNG image bytes
    """
    from PIL import Image
    
    # Load images
    images = {}
    for tf, img_bytes in charts.items():
        img = Image.open(io.BytesIO(img_bytes))
        images[tf] = img
    
    if not images:
        raise ValueError("No charts to combine")
    
    # Get dimensions from first image
    first_img = list(images.values())[0]
    img_width, img_height = first_img.size
    
    # Create 2x2 grid (or smaller if fewer charts)
    num_charts = len(images)
    if num_charts == 1:
        cols, rows = 1, 1
    elif num_charts == 2:
        cols, rows = 2, 1
    elif num_charts <= 4:
        cols, rows = 2, 2
    else:
        cols, rows = 3, 2
    
    # Create combined image
    combined = Image.new('RGB', (img_width * cols, img_height * rows), 'white')
    
    # Paste images in order: 1min, 5min, 15min, 1H
    order = ['1min', '5min', '15min', '1H']
    positions = [(0, 0), (1, 0), (0, 1), (1, 1)]
    
    for i, tf in enumerate(order):
        if tf in images and i < len(positions):
            col, row = positions[i]
            combined.paste(images[tf], (col * img_width, row * img_height))
    
    # Save to bytes
    buf = io.BytesIO()
    combined.save(buf, format='PNG')
    buf.seek(0)
    return buf.read()


print("✓ Multi-Timeframe Chart Generator defined")
print("\nTimeframe Configuration:")
print("-" * 50)
for tf, config in TIMEFRAME_CONFIG.items():
    print(f"  {tf:6s}: {config['bars']} bars = {config['description']}")

✓ Multi-Timeframe Chart Generator defined

Timeframe Configuration:
--------------------------------------------------
  1min  : 200 bars = ~3.3 hours
  5min  : 200 bars = ~16.7 hours
  15min : 200 bars = ~50 hours (2 days)
  1H    : 200 bars = ~8.3 days


In [5]:
# Cell 5: Backtest Adapter for Technical Analyst
# Wraps the actual TechnicalAnalystAgent for backtest usage (no database needed)

class TechnicalAnalystBacktest:
    """
    Backtest adapter for Technical Analyst.
    Uses the actual agent's cascade logic but with DataFrame input instead of database.
    
    When you modify backend/agents/technicalanalyst_agent.py, the core logic changes.
    This adapter just provides the data interface for backtesting.
    """
    
    def __init__(self):
        # Import the actual services used by TechnicalAnalystAgent
        try:
            from services.timeframe_cascade import get_timeframe_cascade
            self.cascade_analyzer = get_timeframe_cascade()
            self.use_actual_cascade = True
            print("  ✓ Using actual timeframe_cascade service from backend")
        except Exception as e:
            print(f"  ⚠ Could not load timeframe_cascade: {e}")
            print("    Using simplified fallback logic")
            self.cascade_analyzer = None
            self.use_actual_cascade = False
    
    def analyze(self, df: pd.DataFrame, idx: int, market_structure: MarketStructure) -> Dict:
        """
        Analyze market data using actual backend logic.
        
        Args:
            df: DataFrame with OHLCV data
            idx: Current bar index (no look-ahead)
            market_structure: Current market structure state
        
        Returns:
            Analysis dict with signal, confidence, reasoning
        """
        if idx < 20:
            return {
                'signal': 'NEUTRAL',
                'confidence': 0.0,
                'reasoning': 'Insufficient data for analysis',
                'confluence_score': 0
            }
        
        data = df.iloc[:idx+1].copy()
        
        if self.use_actual_cascade and self.cascade_analyzer:
            # Use actual cascade analyzer from backend
            try:
                # Prepare timeframe data (simulated aggregation for backtest)
                timeframe_data = self._prepare_timeframe_data(data)
                
                # Run actual cascade analysis
                analysis = self.cascade_analyzer.analyze_all_timeframes(timeframe_data)
                
                return {
                    'signal': analysis.get('signal', 'NEUTRAL'),
                    'confidence': analysis.get('confidence', 0.0),
                    'reasoning': analysis.get('reasoning', ''),
                    'confluence_score': analysis.get('confluence_score', 0),
                    'lead_timeframe': analysis.get('lead_timeframe'),
                    'decision_case': analysis.get('decision_case'),
                    'entry_timing': analysis.get('entry_timing', {})
                }
            except Exception as e:
                print(f"    [Technical] Cascade error: {e}, using fallback")
                return self._fallback_analyze(data, market_structure)
        else:
            return self._fallback_analyze(data, market_structure)
    
    def _prepare_timeframe_data(self, df: pd.DataFrame) -> Dict:
        """Prepare multi-timeframe data from 1-min DataFrame"""
        timeframe_data = {'1m': df}
        
        # Aggregate to higher timeframes
        for tf, period in [('5m', 5), ('15m', 15), ('1h', 60)]:
            if len(df) >= period * 20:  # Need enough bars
                agg_df = self._aggregate_ohlcv(df, period)
                if not agg_df.empty:
                    timeframe_data[tf] = agg_df
        
        return timeframe_data
    
    def _aggregate_ohlcv(self, df: pd.DataFrame, period: int) -> pd.DataFrame:
        """Aggregate 1-minute data to higher timeframe"""
        df = df.copy()
        df['Timestamp'] = pd.to_datetime(df['Timestamp'])
        df = df.set_index('Timestamp')
        
        try:
            resampled = df.resample(f'{period}min').agg({
                'Open': 'first',
                'High': 'max',
                'Low': 'min',
                'Close': 'last',
                'Volume': 'sum'
            }).dropna()
            return resampled.reset_index()
        except:
            return pd.DataFrame()
    
    def _fallback_analyze(self, data: pd.DataFrame, market_structure: MarketStructure) -> Dict:
        """Simplified fallback analysis when cascade service unavailable"""
        current = data.iloc[-1]
        
        confluence_factors = []
        signal_scores = {'LONG': 0, 'SHORT': 0}
        
        # Trend analysis
        if market_structure.trend == 'bullish':
            signal_scores['LONG'] += 1
            confluence_factors.append('Bullish trend (HH/HL)')
        elif market_structure.trend == 'bearish':
            signal_scores['SHORT'] += 1
            confluence_factors.append('Bearish trend (LH/LL)')
        
        # Price at Order Block
        active_obs = [ob for ob in market_structure.order_blocks if not ob.mitigated][-3:]
        for ob in active_obs:
            if ob.ob_type == 'bullish' and current['Low'] <= ob.high <= current['High']:
                signal_scores['LONG'] += 2
                confluence_factors.append(f'At bullish OB ({ob.low:.2f}-{ob.high:.2f})')
            elif ob.ob_type == 'bearish' and current['Low'] <= ob.low <= current['High']:
                signal_scores['SHORT'] += 2
                confluence_factors.append(f'At bearish OB ({ob.low:.2f}-{ob.high:.2f})')
        
        # Unfilled FVG
        active_fvgs = [fvg for fvg in market_structure.fvgs if not fvg.filled][-3:]
        for fvg in active_fvgs:
            if fvg.fvg_type == 'bullish' and current['Close'] > fvg.high:
                signal_scores['LONG'] += 1
                confluence_factors.append('Above bullish FVG')
            elif fvg.fvg_type == 'bearish' and current['Close'] < fvg.low:
                signal_scores['SHORT'] += 1
                confluence_factors.append('Below bearish FVG')
        
        # Calculate final signal
        if signal_scores['LONG'] > signal_scores['SHORT'] and signal_scores['LONG'] >= 3:
            signal = 'LONG'
            confidence = min(0.9, 0.4 + (signal_scores['LONG'] * 0.1))
        elif signal_scores['SHORT'] > signal_scores['LONG'] and signal_scores['SHORT'] >= 3:
            signal = 'SHORT'
            confidence = min(0.9, 0.4 + (signal_scores['SHORT'] * 0.1))
        else:
            signal = 'NEUTRAL'
            confidence = 0.3
        
        reasoning = f"Trend: {market_structure.trend}. " + "; ".join(confluence_factors[:3]) if confluence_factors else "No confluence"
        
        return {
            'signal': signal,
            'confidence': confidence,
            'reasoning': reasoning,
            'confluence_score': len(confluence_factors),
            'confluence_factors': confluence_factors
        }

print("✓ Technical Analyst Backtest Adapter defined")
tech_analyst = TechnicalAnalystBacktest()

✓ Technical Analyst Backtest Adapter defined
  ✓ Using actual timeframe_cascade service from backend


In [6]:
# Cell 6: Backtest Adapter for Visual Analyst (MULTI-TIMEFRAME - SEPARATE IMAGES)
# Uses the shared VisualAnalysisService from backend/services/analysis/visual_analysis.py
# Sends 4 SEPARATE full-size charts to the LLM for better pattern recognition

from services.analysis.visual_analysis import (
    VisualAnalysisService,
    create_visual_analysis_service,
    DEFAULT_VISUAL_PROMPT
)

# Enhanced prompt for multi-timeframe analysis with SEPARATE images
MULTI_TIMEFRAME_VISUAL_PROMPT = """You are an expert technical analyst specializing in multi-timeframe chart analysis for USGOLD (spot gold).

You are viewing 4 SEPARATE chart images, one for each timeframe:
1. **1-minute chart** (~3.3 hours of data, 200 bars) - Entry timing
2. **5-minute chart** (~16.7 hours of data, 200 bars) - Short-term structure
3. **15-minute chart** (~50 hours / 2 days of data, 200 bars) - Intermediate trend
4. **1-hour chart** (~8.3 days of data, 200 bars) - Primary trend direction

Each image shows the FULL chart at high resolution for detailed pattern analysis.

Your task is to perform MULTI-TIMEFRAME ANALYSIS:

1. **Higher Timeframe Trend (1H)**: What is the dominant trend direction?
   - Look for swing highs/lows, trend structure (HH/HL for uptrend, LH/LL for downtrend)
   - This sets the PRIMARY BIAS for trades

2. **Intermediate Confirmation (15min, 5min)**: Does the intermediate timeframe confirm?
   - Look for trend alignment with 1H
   - Identify key support/resistance levels
   - Look for FVGs (Fair Value Gaps) and Order Blocks

3. **Entry Timeframe (1min)**: Is there a valid entry setup?
   - Look for price action patterns at key levels
   - Identify FVGs, Order Blocks, candlestick patterns
   - Entry should align with higher timeframe bias

4. **Pattern Recognition** (across all timeframes):
   - Classic patterns: Head & Shoulders, Double Top/Bottom, Triangles, Flags, Wedges
   - Smart Money Concepts: FVG, Order Blocks, Breaker Blocks, Liquidity Sweeps
   - Candlestick patterns: Engulfing, Doji, Hammer, Shooting Star

5. **Timeframe Confluence**: Do multiple timeframes agree?
   - Strong signal = 3-4 timeframes aligned
   - Medium signal = 2 timeframes aligned
   - Weak signal = timeframes conflicting

**TRADING RULES:**
- Only BUY if 1H shows uptrend AND lower timeframes confirm with bullish structure
- Only SELL if 1H shows downtrend AND lower timeframes confirm with bearish structure
- NEUTRAL if timeframes conflict, no clear setup, or approaching major resistance/support
- Confidence > 70% required for BUY/SELL signal

Return your analysis in this EXACT JSON format:
```json
{
  "signal": "BUY|SELL|NEUTRAL",
  "confidence": 85,
  "timeframe_analysis": {
    "1H": {"trend": "bullish|bearish|neutral", "structure": "HH/HL|LH/LL|range", "notes": "..."},
    "15min": {"trend": "bullish|bearish|neutral", "notes": "..."},
    "5min": {"trend": "bullish|bearish|neutral", "notes": "..."},
    "1min": {"trend": "bullish|bearish|neutral", "entry_setup": "yes|no", "notes": "..."}
  },
  "patterns": ["FVG at 2650-2655 on 5min", "bullish order block on 1H at 2640"],
  "support_levels": [2640, 2620],
  "resistance_levels": [2680, 2700],
  "trend": "bullish|bearish|neutral",
  "trend_strength": "strong|moderate|weak",
  "confluence_score": 4,
  "reasoning": "1H showing clear uptrend with HH/HL structure. 15min confirming with bullish FVG at 2655. 5min pullback to order block complete. 1min showing bullish engulfing at support - valid entry. 4/4 timeframes bullish.",
  "entry_zone": {"low": 2648, "high": 2652},
  "stop_loss": 2640,
  "take_profit": 2680,
  "warnings": ["Watch for resistance at 2680", "News event in 2 hours"]
}
```

Provide only the JSON output, no additional text."""


class VisualAnalystBacktest:
    """
    Backtest adapter for Visual Analyst with MULTI-TIMEFRAME support.
    
    Generates 4 separate charts (1min, 5min, 15min, 1H) and sends them
    individually to the LLM for higher resolution analysis.
    """
    
    def __init__(self, llm_client, use_multi_timeframe: bool = True):
        self.llm_client = llm_client
        self.use_multi_timeframe = use_multi_timeframe
        self.call_count = 0
        self.total_cost = 0.0
        
        # Use multi-timeframe prompt
        prompt = MULTI_TIMEFRAME_VISUAL_PROMPT if use_multi_timeframe else DEFAULT_VISUAL_PROMPT
        
        if llm_client:
            self.analysis_service = create_visual_analysis_service(
                llm_client=llm_client,
                prompt_template=prompt
            )
            print("  ✓ Using MULTI-TIMEFRAME VisualAnalysisService (SEPARATE IMAGES)")
            print(f"    Timeframes: {list(TIMEFRAME_CONFIG.keys())}")
            print(f"    Each chart sent at FULL resolution for better pattern detection")
        else:
            self.analysis_service = None
            print("  ⚠ No LLM client - visual analysis will use fallback")
    
    def analyze(self, df: pd.DataFrame, idx: int, market_structure: MarketStructure) -> Dict:
        """
        Analyze charts using multi-timeframe approach with SEPARATE images.
        
        Generates 4 individual charts (1min, 5min, 15min, 1H) and sends
        each as a separate high-resolution image to the LLM.
        """
        # Need enough data for higher timeframes
        min_bars = 500  # Minimum for reasonable multi-timeframe analysis
        
        if idx < min_bars:
            return {
                'signal': 'NEUTRAL',
                'confidence': 0.0,
                'reasoning': f'Insufficient data for multi-timeframe analysis (need {min_bars}+ bars)',
                'patterns': [],
                'timeframe_analysis': {}
            }
        
        if self.analysis_service is None:
            return self._fallback_analysis(market_structure)
        
        try:
            print(f"    📊 Generating multi-timeframe charts (SEPARATE full-size images)...")
            
            # Generate charts for all timeframes
            charts = generate_multi_timeframe_charts(df, idx, symbol='USGOLD')
            
            if len(charts) < 2:
                print(f"    ⚠️ Only {len(charts)} timeframe(s) available, need at least 2")
                return self._fallback_analysis(market_structure)
            
            # Build market context
            current = df.iloc[idx]
            market_context = {
                'symbol': 'USGOLD',
                'price': current['Close'],
                'timeframes_included': list(charts.keys()),
                'market_structure': market_structure.get_context_string()
            }
            
            # Use the new multi-image analysis method
            print(f"    🖼️ Sending {len(charts)} separate full-size charts to LLM...")
            analysis = self.analysis_service.analyze_multi_timeframe(
                images=charts,
                market_context=market_context
            )
            
            self.call_count += 1
            self.total_cost += 0.03  # Higher cost for multiple images
            
            print(f"    [Visual Analyst LLM] Signal: {analysis['signal']} ({analysis['confidence']:.0%})")
            if 'confluence_score' in analysis:
                print(f"    [Visual Analyst LLM] Confluence: {analysis['confluence_score']}/4 timeframes")
            
            return analysis
            
        except Exception as e:
            print(f"    [Visual Analyst] Error: {e}")
            import traceback
            traceback.print_exc()
            return self._fallback_analysis(market_structure)
    
    def _fallback_analysis(self, market_structure: MarketStructure) -> Dict:
        return {
            'signal': 'NEUTRAL',
            'confidence': 0.0,
            'reasoning': 'LLM unavailable - using fallback',
            'patterns': [],
            'trend': market_structure.trend,
            'timeframe_analysis': {}
        }

print("✓ Visual Analyst Backtest Adapter defined (MULTI-TIMEFRAME - SEPARATE IMAGES)")
visual_analyst = VisualAnalystBacktest(llm_client, use_multi_timeframe=True)

✓ Visual Analyst Backtest Adapter defined (MULTI-TIMEFRAME - SEPARATE IMAGES)
  ✓ Using MULTI-TIMEFRAME VisualAnalysisService (SEPARATE IMAGES)
    Timeframes: ['1min', '5min', '15min', '1H']
    Each chart sent at FULL resolution for better pattern detection


In [7]:
# Cell 7: Backtest Adapter for Meta-Agent (SHARED SERVICE)
# Uses the shared MetaDecisionService from backend/services/analysis/meta_decision.py
# Changes to that file will be reflected here!

from services.analysis.meta_decision import (
    MetaDecisionService,
    create_meta_decision_service,
    DEFAULT_META_PROMPT
)


class MetaAgentBacktest:
    """
    Backtest adapter for Meta-Agent.
    Uses the SHARED MetaDecisionService - identical logic to production.
    
    When you modify backend/services/analysis/meta_decision.py, the logic changes here too.
    """
    
    def __init__(self, llm_client):
        self.llm_client = llm_client
        self.call_count = 0
        self.total_cost = 0.0
        
        # Default weights (actual agent loads from weight_manager)
        self.weights = {'visual': 0.35, 'technical': 0.50, 'sentiment': 0.15}
        
        # Try to load weights from weight_manager
        try:
            from services.weight_manager import get_weight_manager
            weight_manager = get_weight_manager()
            self.weights = weight_manager.get_current_weights()
            print(f"  ✓ Loaded weights from weight_manager: {self.weights}")
        except Exception as e:
            print(f"  ⚠ Could not load weights: {e}, using defaults")
        
        # Use the shared decision service (same as production agent)
        self.decision_service = create_meta_decision_service(
            llm_client=llm_client,
            weights=self.weights,
            prompt_template=DEFAULT_META_PROMPT
        )
        print("  ✓ Using shared MetaDecisionService from backend")
    
    def make_decision(
        self,
        visual_analysis: Dict,
        technical_analysis: Dict,
        market_structure: MarketStructure,
        current_price: float
    ) -> Dict:
        """
        Make trading decision using shared MetaDecisionService.
        Identical logic to production MetaAgent.
        """
        # Prepare inputs for shared service
        visual_input = {
            'signal': visual_analysis.get('signal', 'NEUTRAL'),
            'confidence': visual_analysis.get('confidence', 0.0)
        }
        technical_input = {
            'signal': technical_analysis.get('signal', 'NEUTRAL'),
            'confidence': technical_analysis.get('confidence', 0.0)
        }
        # Skip sentiment in backtest
        sentiment_input = {'signal': 'PASS', 'confidence': 0.5}
        
        market_context = {
            'symbol': 'USGOLD',
            'market_structure': market_structure.get_context_string(),
            'current_price': current_price
        }
        
        # Use shared decision service (IDENTICAL to production)
        decision = self.decision_service.make_decision(
            visual_analysis=visual_input,
            technical_analysis=technical_input,
            sentiment_analysis=sentiment_input,
            market_context=market_context,
            current_price=current_price,
            use_llm=(self.llm_client is not None)
        )
        
        if self.llm_client:
            self.call_count = self.decision_service.call_count
            self.total_cost = self.call_count * 0.005
        
        print(f"    [Meta-Agent] Decision: {decision['decision']} "
              f"(conf: {decision['confidence']:.0%}, score: {decision.get('weighted_score', 0):.2f})")
        
        return decision

print("✓ Meta-Agent Backtest Adapter defined (uses shared service)")
meta_agent = MetaAgentBacktest(llm_client)

✓ Meta-Agent Backtest Adapter defined (uses shared service)
  ⚠ Could not load weights: Connection pool not initialized. Call create_pool() first., using defaults
  ✓ Using shared MetaDecisionService from backend


In [8]:
# Cell 8: Risk Manager, Execution, and Trade Manager (Backtest versions)
# These are simpler and don't need database, so we use lightweight implementations

@dataclass
class Trade:
    trade_id: int
    symbol: str
    direction: str
    entry_price: float
    entry_time: datetime
    stop_loss: float
    take_profit: float
    position_size: float
    status: str = 'OPEN'
    exit_price: Optional[float] = None
    exit_time: Optional[datetime] = None
    pnl: float = 0.0
    exit_reason: str = ''


class RiskManagerBacktest:
    """
    Backtest version of Risk Manager.
    Uses same hard limits as actual riskmanager_agent.py
    """
    
    def __init__(self, initial_balance: float = 10000.0):
        self.initial_balance = initial_balance
        self.current_balance = initial_balance
        
        # Hard limits (same as actual agent)
        self.max_risk_per_trade = 0.02
        self.max_daily_loss = 0.06
        self.max_drawdown = 0.15
        self.max_positions = 2
        
        self.daily_pnl = 0.0
        self.open_positions = 0
    
    def validate_trade(self, decision: Dict, entry_price: float, atr: float) -> Dict:
        """Validate trade and calculate position sizing"""
        direction = decision['decision']
        confidence = decision.get('confidence', 0.5)
        
        # Hard limit checks
        if self.open_positions >= self.max_positions:
            return {'status': 'REJECTED', 'reason': f'Max positions ({self.max_positions})'}
        
        if self.daily_pnl < 0 and abs(self.daily_pnl) / self.initial_balance >= self.max_daily_loss:
            return {'status': 'REJECTED', 'reason': 'Daily loss limit'}
        
        drawdown = (self.initial_balance - self.current_balance) / self.initial_balance
        if drawdown >= self.max_drawdown:
            return {'status': 'REJECTED', 'reason': 'Max drawdown limit'}
        
        # Calculate SL/TP (same logic as actual agent)
        stop_distance = 1.5 * atr
        if direction == 'LONG':
            stop_loss = entry_price - stop_distance
            take_profit = entry_price + (stop_distance * 2)
        else:
            stop_loss = entry_price + stop_distance
            take_profit = entry_price - (stop_distance * 2)
        
        # Position sizing (confidence-based)
        risk_pct = self.max_risk_per_trade * confidence
        risk_dollars = self.current_balance * risk_pct
        position_size = risk_dollars / stop_distance
        
        return {
            'status': 'APPROVED',
            'trade_parameters': {
                'direction': direction,
                'entry_price': entry_price,
                'stop_loss': stop_loss,
                'take_profit': take_profit,
                'position_size': position_size,
                'risk_dollars': risk_dollars,
                'risk_pct': risk_pct,
                'risk_reward_ratio': 2.0
            }
        }
    
    def update_balance(self, pnl: float):
        self.current_balance += pnl
        self.daily_pnl += pnl


class ExecutionAgentBacktest:
    def __init__(self):
        self.trade_counter = 0
    
    def execute_trade(self, trade_params: Dict, symbol: str, timestamp: datetime) -> Trade:
        self.trade_counter += 1
        return Trade(
            trade_id=self.trade_counter,
            symbol=symbol,
            direction=trade_params['direction'],
            entry_price=trade_params['entry_price'],
            entry_time=timestamp,
            stop_loss=trade_params['stop_loss'],
            take_profit=trade_params['take_profit'],
            position_size=trade_params['position_size']
        )


class TradeManagerBacktest:
    def __init__(self):
        self.active_trades: Dict[int, Trade] = {}
        self.closed_trades: List[Trade] = []
        self.max_trade_time = 40  # minutes
    
    def add_trade(self, trade: Trade):
        self.active_trades[trade.trade_id] = trade
    
    def check_exits(self, current_price: float, current_time: datetime) -> List[Tuple[Trade, str]]:
        exits = []
        
        for trade_id, trade in list(self.active_trades.items()):
            exit_reason = None
            exit_price = current_price
            
            # Stop loss
            if trade.direction == 'LONG' and current_price <= trade.stop_loss:
                exit_reason = 'STOP_LOSS'
                exit_price = trade.stop_loss
            elif trade.direction == 'SHORT' and current_price >= trade.stop_loss:
                exit_reason = 'STOP_LOSS'
                exit_price = trade.stop_loss
            
            # Take profit
            if trade.direction == 'LONG' and current_price >= trade.take_profit:
                exit_reason = 'TAKE_PROFIT'
                exit_price = trade.take_profit
            elif trade.direction == 'SHORT' and current_price <= trade.take_profit:
                exit_reason = 'TAKE_PROFIT'
                exit_price = trade.take_profit
            
            # Time limit
            time_in_trade = (current_time - trade.entry_time).total_seconds() / 60
            if time_in_trade > self.max_trade_time:
                exit_reason = 'TIME_LIMIT'
            
            if exit_reason:
                trade.exit_price = exit_price
                trade.exit_time = current_time
                trade.exit_reason = exit_reason
                trade.status = 'CLOSED'
                
                if trade.direction == 'LONG':
                    trade.pnl = (exit_price - trade.entry_price) * trade.position_size
                else:
                    trade.pnl = (trade.entry_price - exit_price) * trade.position_size
                
                exits.append((trade, exit_reason))
                del self.active_trades[trade_id]
                self.closed_trades.append(trade)
        
        return exits

print("✓ Risk Manager, Execution Agent, and Trade Manager defined")

✓ Risk Manager, Execution Agent, and Trade Manager defined


In [9]:
# Cell 9: Backtest Orchestrator
# Coordinates all agents and runs the simulation

class BacktestOrchestrator:
    """
    Coordinates all agents for backtesting.
    Uses actual backend agent logic via backtest adapters.
    """
    
    def __init__(self, llm_client, initial_balance: float = 10000.0):
        # Backtest adapters (linked to actual backend agents)
        self.technical_analyst = TechnicalAnalystBacktest()
        self.visual_analyst = VisualAnalystBacktest(llm_client)
        self.meta_agent = MetaAgentBacktest(llm_client)
        
        # Backtest versions of other agents
        self.risk_manager = RiskManagerBacktest(initial_balance)
        self.execution_agent = ExecutionAgentBacktest()
        self.trade_manager = TradeManagerBacktest()
        
        # Market structure tracker
        self.market_structure = MarketStructure(lookback=5)
        
        # Communication log
        self.message_log: List[Dict] = []
        
        # Stats
        self.total_signals = 0
        self.approved_trades = 0
        self.rejected_trades = 0
    
    def log_message(self, sender: str, recipient: str, msg_type: str, content: str):
        self.message_log.append({
            'sender': sender,
            'recipient': recipient,
            'type': msg_type,
            'content': content
        })
    
    def process_bar(self, df: pd.DataFrame, idx: int, symbol: str = 'USGOLD') -> Dict:
        """Process a single bar through the agent pipeline"""
        current = df.iloc[idx]
        timestamp = current['Timestamp']
        current_price = current['Close']
        
        result = {
            'bar_idx': idx,
            'timestamp': timestamp,
            'price': current_price,
            'agents': {},
            'trade_action': None,
            'exits': []
        }
        
        # 1. Update market structure
        self.market_structure.process_bar(df, idx)
        
        # 2. Check exits for open positions
        exits = self.trade_manager.check_exits(current_price, timestamp)
        for trade, reason in exits:
            result['exits'].append({'trade_id': trade.trade_id, 'reason': reason, 'pnl': trade.pnl})
            self.risk_manager.update_balance(trade.pnl)
            self.risk_manager.open_positions -= 1
            self.log_message('TradeManager', 'Execution', 'TRADE_CLOSED',
                           f'Trade #{trade.trade_id}: {reason}, PnL: ${trade.pnl:.2f}')
        
        # Only analyze every N bars to simulate trigger events and reduce LLM costs
        if idx % 10 != 0 or idx < 20:
            return result
        
        print(f"\n{'='*60}")
        print(f"Bar {idx} | {timestamp} | Price: ${current_price:.2f}")
        print(f"{'='*60}")
        
        # 3. Technical Analyst (uses actual backend cascade logic)
        print("\n  [Technical Analyst] Analyzing...")
        tech_analysis = self.technical_analyst.analyze(df, idx, self.market_structure)
        result['agents']['technical'] = tech_analysis
        print(f"    Signal: {tech_analysis['signal']} ({tech_analysis['confidence']:.0%})")
        print(f"    Reasoning: {tech_analysis['reasoning']}")
        
        self.log_message('TechnicalAnalyst', 'MetaAgent', 'ANALYSIS_RESULT',
                        f"{tech_analysis['signal']} ({tech_analysis['confidence']:.0%})")
        
        # 4. Visual Analyst (LLM-based with actual backend prompts)
        print("\n  [Visual Analyst] Analyzing chart with LLM Vision...")
        visual_analysis = self.visual_analyst.analyze(df, idx, self.market_structure)
        result['agents']['visual'] = visual_analysis
        print(f"    Signal: {visual_analysis['signal']} ({visual_analysis['confidence']:.0%})")
        print(f"    Reasoning: {visual_analysis.get('reasoning', 'N/A')}")
        
        self.log_message('VisualAnalyst', 'MetaAgent', 'ANALYSIS_RESULT',
                        f"{visual_analysis['signal']} ({visual_analysis['confidence']:.0%})")
        
        # 5. Meta-Agent Decision (LLM-based with actual backend prompts)
        print("\n  [Meta-Agent] Making decision with LLM...")
        meta_decision = self.meta_agent.make_decision(
            visual_analysis, tech_analysis, self.market_structure, current_price
        )
        result['agents']['meta'] = meta_decision
        print(f"    Decision: {meta_decision['decision']}")
        print(f"    Reasoning: {meta_decision.get('reasoning', 'N/A')}")
        
        if meta_decision['decision'] != 'PASS':
            self.total_signals += 1
            
            self.log_message('MetaAgent', 'RiskManager', 'TRADE_SIGNAL',
                           f"{meta_decision['decision']} (conf: {meta_decision['confidence']:.0%})")
            
            # 6. Risk Manager (uses actual backend limits)
            print("\n  [Risk Manager] Validating trade...")
            atr = self.market_structure.get_atr(df, idx)
            risk_result = self.risk_manager.validate_trade(meta_decision, current_price, atr)
            result['agents']['risk'] = risk_result
            
            if risk_result['status'] == 'APPROVED':
                self.approved_trades += 1
                params = risk_result['trade_parameters']
                print(f"    ✅ APPROVED: {params['direction']} @ ${current_price:.2f}")
                print(f"       SL: ${params['stop_loss']:.2f}, TP: ${params['take_profit']:.2f}")
                
                self.log_message('RiskManager', 'Execution', 'TRADE_APPROVED',
                               f"{params['direction']} @ ${current_price:.2f}")
                
                # 7. Execute trade
                trade = self.execution_agent.execute_trade(params, symbol, timestamp)
                self.trade_manager.add_trade(trade)
                self.risk_manager.open_positions += 1
                
                result['trade_action'] = {'action': 'OPENED', 'trade': trade}
                
                print(f"\n  ⚡ TRADE OPENED: #{trade.trade_id}")
                self.log_message('Execution', 'TradeManager', 'TRADE_OPENED',
                               f"Trade #{trade.trade_id}")
            else:
                self.rejected_trades += 1
                print(f"    ❌ REJECTED: {risk_result['reason']}")
                self.log_message('RiskManager', 'MetaAgent', 'TRADE_REJECTED',
                               risk_result['reason'])
        else:
            print("\n  → No trade (PASS)")
        
        # Print balance
        print(f"\n  💰 Balance: ${self.risk_manager.current_balance:.2f}")
        
        return result
    
    def get_llm_costs(self) -> Dict:
        return {
            'visual_analyst': {
                'calls': self.visual_analyst.call_count,
                'estimated_cost': self.visual_analyst.total_cost
            },
            'meta_agent': {
                'calls': self.meta_agent.call_count,
                'estimated_cost': self.meta_agent.total_cost
            },
            'total_cost': self.visual_analyst.total_cost + self.meta_agent.total_cost
        }

print("✓ Backtest Orchestrator defined")

✓ Backtest Orchestrator defined


In [10]:
# Cell 10: Load Data from CSV Snapshot

def load_csv_data(filepath: str) -> pd.DataFrame:
    """Load OHLCV data from CSV snapshot"""
    df = pd.read_csv(filepath)
    
    # Standardize column names
    column_mapping = {
        'timestamp': 'Timestamp', 'time': 'Timestamp', 'date': 'Timestamp',
        'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 'volume': 'Volume'
    }
    df = df.rename(columns={k: v for k, v in column_mapping.items() if k in df.columns})
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    
    for col in ['Open', 'High', 'Low', 'Close']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    if 'Volume' not in df.columns:
        df['Volume'] = 0
    
    df = df.sort_values('Timestamp').reset_index(drop=True)
    
    print(f"✓ Loaded {len(df)} bars from {filepath}")
    print(f"  Date range: {df['Timestamp'].min()} to {df['Timestamp'].max()}")
    print(f"  Price range: ${df['Low'].min():.2f} - ${df['High'].max():.2f}")
    
    return df

# Load Jan 20-30, 2026 snapshot
df = load_csv_data('../data/snapshots/USGOLD_20260120_20260130.csv')

print(f"\nFirst 5 bars:")
print(df.head())
print(f"\nLast 5 bars:")
print(df.tail())

✓ Loaded 11799 bars from ../data/snapshots/USGOLD_20260120_20260130.csv
  Date range: 2026-01-20 00:00:00 to 2026-01-30 16:00:00
  Price range: $4659.21 - $5595.53

First 5 bars:
            Timestamp     Open     High      Low    Close  Volume
0 2026-01-20 00:00:00  4664.61  4666.12  4663.81  4665.72      -1
1 2026-01-20 00:01:00  4665.72  4665.77  4664.31  4664.77      -1
2 2026-01-20 00:02:00  4664.77  4664.92  4663.61  4663.86      -1
3 2026-01-20 00:03:00  4663.86  4664.87  4663.66  4664.11      -1
4 2026-01-20 00:04:00  4664.11  4664.57  4663.66  4664.11      -1

Last 5 bars:
                Timestamp     Open     High      Low    Close  Volume
11794 2026-01-30 15:56:00  5049.53  5054.65  5046.10  5053.83      -1
11795 2026-01-30 15:57:00  5053.83  5054.32  5042.90  5047.07      -1
11796 2026-01-30 15:58:00  5047.07  5053.41  5044.05  5051.79      -1
11797 2026-01-30 15:59:00  5051.79  5057.45  5048.38  5050.66      -1
11798 2026-01-30 16:00:00  5050.66  5051.86  5045.27  5049.87

In [11]:
# Cell 11: VISUAL ANALYST DEBUG - See the SEPARATE charts the LLM analyzes
# Each timeframe is displayed at FULL resolution (not a scaled grid)

def visualize_multi_timeframe_analysis(df: pd.DataFrame, idx: int, llm_client, show_charts: bool = True):
    """
    Generate and display the multi-timeframe charts that Visual Analyst sees.
    Each chart is shown separately at FULL resolution for better detail.
    
    Args:
        df: DataFrame with 1-min OHLCV data
        idx: Bar index to analyze (needs 500+ bars for good results)
        llm_client: LLM client for analysis
        show_charts: Whether to display charts inline
    
    Returns:
        Analysis result dict
    """
    from IPython.display import display, Image as IPImage
    from services.analysis.market_structure import create_market_structure_analyzer
    
    print(f"{'='*70}")
    print(f"MULTI-TIMEFRAME VISUAL ANALYST DEBUG - Bar {idx}")
    print(f"{'='*70}")
    
    current = df.iloc[idx]
    print(f"\nTimestamp: {current['Timestamp']}")
    print(f"Price: ${current['Close']:.2f}")
    
    # Check minimum data requirement
    if idx < 500:
        print(f"\n⚠️ Need at least 500 bars for multi-timeframe analysis (have {idx})")
        print("   Try a higher bar index (e.g., 1000, 5000)")
        return None
    
    # 1. Generate multi-timeframe charts
    print(f"\n📊 Generating multi-timeframe charts...")
    print("-" * 70)
    charts = generate_multi_timeframe_charts(df, idx, symbol='USGOLD')
    print("-" * 70)
    
    if len(charts) < 2:
        print("❌ Not enough timeframes generated")
        return None
    
    # 2. Show each chart SEPARATELY at full resolution
    if show_charts:
        print(f"\n📈 FULL-RESOLUTION CHARTS (Each sent separately to LLM):")
        print("=" * 70)
        for tf_name in ['1H', '15min', '5min', '1min']:  # Show higher TF first
            if tf_name in charts:
                config = TIMEFRAME_CONFIG[tf_name]
                print(f"\n{'─'*70}")
                print(f"📊 {tf_name} CHART ({config['bars']} bars = {config['description']})")
                print(f"{'─'*70}")
                display(IPImage(data=charts[tf_name]))
        print("=" * 70)
    
    # 3. Build market structure context
    print(f"\n🔍 Building market structure context...")
    market_structure = create_market_structure_analyzer(lookback=5)
    for i in range(max(0, idx-200), idx+1):
        market_structure.process_bar(df, i)
    
    print(f"   Trend: {market_structure.trend}")
    print(f"   Swing Points: {len(market_structure.swing_points)}")
    print(f"   Order Blocks: {len([ob for ob in market_structure.order_blocks if not ob.mitigated])} active")
    print(f"   FVGs: {len([fvg for fvg in market_structure.fvgs if not fvg.filled])} unfilled")
    
    # 4. Call the Visual Analyst LLM with SEPARATE images
    if llm_client:
        print(f"\n🤖 CALLING LLM VISION API with {len(charts)} SEPARATE full-size images...")
        print(f"   Model: {llm_client.model}")
        print(f"   Images: {list(charts.keys())}")
        
        analysis_service = create_visual_analysis_service(
            llm_client=llm_client,
            prompt_template=MULTI_TIMEFRAME_VISUAL_PROMPT
        )
        
        market_context = {
            'symbol': 'USGOLD',
            'price': current['Close'],
            'timeframes_included': list(charts.keys()),
            'market_structure': market_structure.get_context_string()
        }
        
        try:
            # Use new multi-image method
            analysis = analysis_service.analyze_multi_timeframe(
                images=charts,
                market_context=market_context
            )
            
            print(f"\n✅ VISUAL ANALYST MULTI-TIMEFRAME RESPONSE:")
            print("=" * 70)
            print(f"Signal: {analysis['signal']}")
            print(f"Confidence: {analysis['confidence']:.0%}")
            
            # Show timeframe-by-timeframe analysis if available
            if 'timeframe_analysis' in analysis and analysis['timeframe_analysis']:
                print(f"\n📊 Timeframe-by-Timeframe Analysis:")
                for tf in ['1H', '15min', '5min', '1min']:
                    tf_data = analysis.get('timeframe_analysis', {}).get(tf, {})
                    if isinstance(tf_data, dict):
                        trend = tf_data.get('trend', 'N/A')
                        notes = tf_data.get('notes', '')
                        structure = tf_data.get('structure', '')
                        entry = tf_data.get('entry_setup', '')
                        print(f"  {tf:6s}: {trend:8s} {structure} {entry}")
                        if notes:
                            print(f"          {notes}")
            
            if 'confluence_score' in analysis:
                print(f"\n🎯 Confluence Score: {analysis['confluence_score']}/4 timeframes aligned")
            
            print(f"\n💭 Reasoning:")
            print(f"  {analysis.get('reasoning', 'N/A')}")
            
            print(f"\n🔍 Patterns Detected:")
            for pattern in analysis.get('patterns', []):
                print(f"  • {pattern}")
            
            print(f"\n📈 Trend: {analysis.get('trend', 'N/A')} ({analysis.get('trend_strength', 'N/A')})")
            print(f"🟢 Support: {analysis.get('support_levels', [])}")
            print(f"🔴 Resistance: {analysis.get('resistance_levels', [])}")
            
            if analysis.get('entry_zone'):
                ez = analysis['entry_zone']
                print(f"\n🎯 Entry Zone: ${ez.get('low', '?'):.2f} - ${ez.get('high', '?'):.2f}")
            if analysis.get('stop_loss'):
                print(f"🛑 Stop Loss: ${analysis['stop_loss']:.2f}")
            if analysis.get('take_profit'):
                print(f"✅ Take Profit: ${analysis['take_profit']:.2f}")
            
            if analysis.get('warnings'):
                print(f"\n⚠️ Warnings:")
                for warning in analysis['warnings']:
                    print(f"  • {warning}")
            
            print("=" * 70)
            
            return analysis
            
        except Exception as e:
            print(f"\n❌ LLM Error: {e}")
            import traceback
            traceback.print_exc()
            return None
    else:
        print("\n⚠️ No LLM client available")
        return None

print("✓ Multi-Timeframe Visual Analysis Debug function defined")
print("  Each chart is displayed SEPARATELY at full resolution")
print("  Run: visualize_multi_timeframe_analysis(df, idx=5000, llm_client=llm_client)")

✓ Multi-Timeframe Visual Analysis Debug function defined
  Each chart is displayed SEPARATELY at full resolution
  Run: visualize_multi_timeframe_analysis(df, idx=5000, llm_client=llm_client)


In [12]:
# Cell 12: RUN MULTI-TIMEFRAME VISUAL ANALYST DEMO
# This shows the 2x2 chart grid (1min, 5min, 15min, 1H) and the LLM's analysis

# Choose a bar index with enough history (need 500+ for multi-timeframe)
demo_bar_index = 5000  # ~3.5 days into the data

print("🎯 MULTI-TIMEFRAME VISUAL ANALYST DEMO")
print(f"   Analyzing bar {demo_bar_index} of {len(df)} total bars")
print(f"   Timestamp: {df.iloc[demo_bar_index]['Timestamp']}")
print(f"   This will generate 4 charts (1min, 5min, 15min, 1H) and combine them")
print()

# Run the multi-timeframe visual analysis
# Set show_charts=False to only show the combined grid (faster)
analysis_result = visualize_multi_timeframe_analysis(
    df, 
    idx=demo_bar_index, 
    llm_client=llm_client,
    show_charts=False  # Set to True to see individual timeframe charts
)

# Try different bar indices to see different market conditions:
# - 1000: Early in the week
# - 5000: Mid-week
# - 8000: Late week  
# - 11000: End of data period

🎯 MULTI-TIMEFRAME VISUAL ANALYST DEMO
   Analyzing bar 5000 of 11799 total bars
   Timestamp: 2026-01-23 15:02:00
   This will generate 4 charts (1min, 5min, 15min, 1H) and combine them

MULTI-TIMEFRAME VISUAL ANALYST DEBUG - Bar 5000

Timestamp: 2026-01-23 15:02:00
Price: $4946.20

📊 Generating multi-timeframe charts...
----------------------------------------------------------------------
    ✓ 1min: 200 bars (~3.3 hours)
    ✓ 5min: 200 bars (~16.7 hours)
    ✓ 15min: 200 bars (~50 hours (2 days))
    ✓ 1H: 85 bars (~8.3 days)
----------------------------------------------------------------------

🔍 Building market structure context...
   Trend: neutral
   Swing Points: 25
   Order Blocks: 0 active
   FVGs: 4 unfilled

🤖 CALLING LLM VISION API with 4 SEPARATE full-size images...
   Model: gemini-2.0-flash
   Images: ['1min', '5min', '15min', '1H']


Sending 4 charts to LLM: ['1min', '5min', '15min', '1H']
✓ Multi-TF visual analysis: LONG (80%)



✅ VISUAL ANALYST MULTI-TIMEFRAME RESPONSE:
Signal: LONG
Confidence: 80%

💭 Reasoning:
  All timeframes are aligned in a bullish direction. The 1H chart shows a clear uptrend, confirmed by the 15min and 5min charts. The 1min chart is showing a bullish entry setup after a pullback to support. High probability of continuation to the upside.

🔍 Patterns Detected:

📈 Trend: bullish (strong)
🟢 Support: [4930, 4900]
🔴 Resistance: [4960, 4980]

⚠️ Warnings:
  • Potential resistance at 4960
  • Monitor price action closely


In [13]:
# Cell 11: Run Backtest Simulation

def run_backtest(df: pd.DataFrame, llm_client, start_bar: int = 20, end_bar: int = None):
    """
    Run the multi-agent backtest simulation.
    
    Uses actual backend agent logic via backtest adapters.
    Changes to backend/agents/*.py will be reflected here.
    """
    end_bar = end_bar or len(df)
    
    print(f"\n{'#'*60}")
    print(f"# MULTI-AGENT TRADING BACKTEST")
    print(f"# Linked to: backend/agents/*.py")
    print(f"# Bars: {start_bar} to {end_bar}")
    print(f"{'#'*60}\n")
    
    orchestrator = BacktestOrchestrator(llm_client, initial_balance=10000.0)
    
    for idx in range(start_bar, end_bar):
        result = orchestrator.process_bar(df, idx)
    
    # Final summary
    print(f"\n\n{'='*60}")
    print(f"BACKTEST COMPLETE")
    print(f"{'='*60}")
    print(f"Total Signals: {orchestrator.total_signals}")
    print(f"Approved: {orchestrator.approved_trades}")
    print(f"Rejected: {orchestrator.rejected_trades}")
    print(f"Closed Trades: {len(orchestrator.trade_manager.closed_trades)}")
    print(f"")
    print(f"Initial Balance: $10,000.00")
    print(f"Final Balance: ${orchestrator.risk_manager.current_balance:.2f}")
    print(f"Total P&L: ${orchestrator.risk_manager.current_balance - 10000:.2f}")
    
    # Win rate
    if orchestrator.trade_manager.closed_trades:
        wins = sum(1 for t in orchestrator.trade_manager.closed_trades if t.pnl > 0)
        print(f"Win Rate: {wins/len(orchestrator.trade_manager.closed_trades)*100:.1f}%")
    
    # LLM costs
    costs = orchestrator.get_llm_costs()
    print(f"")
    print(f"LLM API Calls:")
    print(f"  Visual Analyst: {costs['visual_analyst']['calls']} calls (~${costs['visual_analyst']['estimated_cost']:.3f})")
    print(f"  Meta-Agent: {costs['meta_agent']['calls']} calls (~${costs['meta_agent']['estimated_cost']:.3f})")
    print(f"  Total Estimated Cost: ~${costs['total_cost']:.3f}")
    
    return orchestrator

# Run backtest
orchestrator = run_backtest(df, llm_client, start_bar=20, end_bar=100)

Missing data for 5m timeframe, skipping
Missing data for 15m timeframe, skipping
Missing data for 1h timeframe, skipping
Missing data for 4h timeframe, skipping



############################################################
# MULTI-AGENT TRADING BACKTEST
# Linked to: backend/agents/*.py
# Bars: 20 to 100
############################################################

  ✓ Using actual timeframe_cascade service from backend
  ✓ Using MULTI-TIMEFRAME VisualAnalysisService (SEPARATE IMAGES)
    Timeframes: ['1min', '5min', '15min', '1H']
    Each chart sent at FULL resolution for better pattern detection
  ⚠ Could not load weights: Connection pool not initialized. Call create_pool() first., using defaults
  ✓ Using shared MetaDecisionService from backend

Bar 20 | 2026-01-20 00:20:00 | Price: $4665.11

  [Technical Analyst] Analyzing...
    Signal: PASS (0%)
    Reasoning: No clear confluence across timeframes or conflicting signals

  [Visual Analyst] Analyzing chart with LLM Vision...
    Signal: NEUTRAL (0%)
    Reasoning: Insufficient data for multi-timeframe analysis (need 500+ bars)

  [Meta-Agent] Making decision with LLM...


✓ Meta decision (LLM): PASS (conf: 8%, score: 0.07)
Missing data for 5m timeframe, skipping
Missing data for 15m timeframe, skipping
Missing data for 1h timeframe, skipping
Missing data for 4h timeframe, skipping


    [Meta-Agent] Decision: PASS (conf: 8%, score: 0.07)
    Decision: PASS
    Reasoning: The decision is PASS because the weighted score (0.075) is below the threshold of 0.7, and the agreement (1.0) doesn't compensate for the low score. The weighted score is calculated as follows: (0.0 * 0.35) + (0.0 * 0.5) + (0.5 * 0.15) = 0.075. Although the agreement is 1.0, indicating full agreement among the analysts (all PASS or NEUTRAL), the low confidence levels from the analysts result in a low weighted score, failing to meet the trading criteria.

  → No trade (PASS)

  💰 Balance: $10000.00

Bar 30 | 2026-01-20 00:30:00 | Price: $4662.36

  [Technical Analyst] Analyzing...
    Signal: PASS (0%)
    Reasoning: No clear confluence across timeframes or conflicting signals

  [Visual Analyst] Analyzing chart with LLM Vision...
    Signal: NEUTRAL (0%)
    Reasoning: Insufficient data for multi-timeframe analysis (need 500+ bars)

  [Meta-Agent] Making decision with LLM...


✓ Meta decision (LLM): PASS (conf: 0%, score: 0.07)
Missing data for 5m timeframe, skipping
Missing data for 15m timeframe, skipping
Missing data for 1h timeframe, skipping
Missing data for 4h timeframe, skipping


...

    [Meta-Agent] Decision: PASS (conf: 0%, score: 0.07)
    Decision: PASS
    Reasoning: All analysts are either PASS or NEUTRAL. The weighted score is only 0.075, which is far below the threshold for initiating a trade. Agreement is 1.0, but this is because no analysts are in disagreement, not because they are all aligned on a specific direction. Therefore, the decision is PASS.

  → No trade (PASS)

  💰 Balance: $10000.00

Bar 80 | 2026-01-20 01:20:00 | Price: $4666.01

  [Technical Analyst] Analyzing...
    Signal: PASS (0%)
    Reasoning: No clear confluence across timeframes or conflicting signals

  [Visual Analyst] Analyzing chart with LLM Vision...
    Signal: NEUTRAL (0%)
    Reasoning: Insufficient data for multi-timeframe analysis (need 500+ bars)

  [Meta-Agent] Making decision with LLM...


✓ Meta decision (LLM): PASS (conf: 0%, score: 0.07)
Missing data for 5m timeframe, skipping
Missing data for 15m timeframe, skipping
Missing data for 1h timeframe, skipping
Missing data for 4h timeframe, skipping


    [Meta-Agent] Decision: PASS (conf: 0%, score: 0.07)
    Decision: PASS
    Reasoning: The weighted score is calculated as follows: (0.0 * 0.35) + (0.0 * 0.5) + (0.5 * 0.15) = 0.075. The agreement is 1.0 because all analysts either signaled PASS or NEUTRAL. Since the weighted score (0.075) is less than 0.7 and the agreement (1.0) does not meet the threshold for reduced confidence trading (agreement >= 0.67 and weighted_score >= 0.6), the decision is PASS.

  → No trade (PASS)

  💰 Balance: $10000.00

Bar 90 | 2026-01-20 01:30:00 | Price: $4670.91

  [Technical Analyst] Analyzing...
    Signal: PASS (0%)
    Reasoning: No clear confluence across timeframes or conflicting signals

  [Visual Analyst] Analyzing chart with LLM Vision...
    Signal: NEUTRAL (0%)
    Reasoning: Insufficient data for multi-timeframe analysis (need 500+ bars)

  [Meta-Agent] Making decision with LLM...


✓ Meta decision (LLM): PASS (conf: 0%, score: 0.07)


    [Meta-Agent] Decision: PASS (conf: 0%, score: 0.07)
    Decision: PASS
    Reasoning: The weighted score is 0.075, which is below the threshold of 0.7. The agreement is 1.0, but the weighted score is the primary factor in this case. The visual analyst signal is NEUTRAL with a confidence of 0.0, the technical analyst signal is PASS with a confidence of 0.0, and the sentiment analyst signal is PASS with a confidence of 0.5.  Therefore, the decision is PASS.

  → No trade (PASS)

  💰 Balance: $10000.00


BACKTEST COMPLETE
Total Signals: 0
Approved: 0
Rejected: 0
Closed Trades: 0

Initial Balance: $10,000.00
Final Balance: $10000.00
Total P&L: $0.00

LLM API Calls:
  Visual Analyst: 0 calls (~$0.000)
  Meta-Agent: 8 calls (~$0.040)
  Total Estimated Cost: ~$0.040


In [14]:
# Cell 12: Trade Log

def print_trade_log(orchestrator):
    trades = orchestrator.trade_manager.closed_trades
    
    if not trades:
        print("No closed trades.")
        return
    
    print(f"\n{'='*80}")
    print("TRADE LOG")
    print(f"{'='*80}")
    print(f"{'ID':>4} | {'Dir':>5} | {'Entry':>10} | {'Exit':>10} | {'P&L':>10} | {'Reason':>12}")
    print(f"{'-'*80}")
    
    total_pnl = 0
    for trade in trades:
        print(f"{trade.trade_id:>4} | {trade.direction:>5} | ${trade.entry_price:>8.2f} | "
              f"${trade.exit_price:>8.2f} | ${trade.pnl:>+9.2f} | {trade.exit_reason:>12}")
        total_pnl += trade.pnl
    
    print(f"{'-'*80}")
    print(f"TOTAL P&L: ${total_pnl:>+.2f}")

print_trade_log(orchestrator)

No closed trades.


## How to Use This Notebook

### Shared Services Architecture

This notebook uses **shared services** that are identical to production. When you modify logic:

| File to Modify | What Changes |
|----------------|--------------|
| `services/analysis/market_structure.py` | Swing point detection, Order Blocks, FVGs |
| `services/analysis/visual_analysis.py` | Visual analysis prompts, parsing, signal logic |
| `services/analysis/meta_decision.py` | Weighted voting, agreement calculation, thresholds |
| `services/timeframe_cascade.py` | Multi-timeframe cascade rules |
| `services/weight_manager.py` | Agent weights |

### One Change = Both Systems Updated

```
┌─────────────────────────────────────────────────────────────┐
│                   SHARED SERVICES                            │
│  services/analysis/visual_analysis.py                        │
│  services/analysis/meta_decision.py                          │
│  services/analysis/market_structure.py                       │
│  services/timeframe_cascade.py                               │
└─────────────────────────────────────────────────────────────┘
                    ▲                    ▲
                    │                    │
          ┌────────┴───────┐   ┌────────┴───────┐
          │  PRODUCTION    │   │   BACKTEST     │
          │  meta_agent.py │   │   notebook.py  │
          │  visual_*.py   │   │                │
          └────────────────┘   └────────────────┘
```

### Workflow
1. Modify shared service in `backend/services/analysis/*.py`
2. Restart kernel (or re-run import cells)
3. Run backtest to see performance
4. Deploy - production uses same logic!

### LLM Costs
- **Visual Analyst**: ~$0.01 per chart analysis (vision API)
- **Meta-Agent**: ~$0.005 per decision (text generation)

### Loading Your Data
```python
df = load_csv_data('../data/gold_1min.csv')
orchestrator = run_backtest(df, llm_client, start_bar=20, end_bar=500)
```

### Environment Setup
Set your API keys in `backend/.env`:
```
LLM_PROVIDER=gemini
GOOGLE_API_KEY=your_key_here
```
or
```
LLM_PROVIDER=claude
ANTHROPIC_API_KEY=your_key_here
```